# **Practical Assignment 6: Clustering Algorithms**
## Real-World Wine Dataset — Unsupervised Learning

Name: Arya M. Bhat

Div: A Batch: A1

Roll no: 10

PRN: 202401040044

Subject: AIML Assignment 6



---


**Objective:** apply multiple clustering algorithms to real-world data, evaluate cluster quality using Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Score, visualize results, and compare methods.

**Dataset:** scikit-learn's Wine recognition dataset (178 samples, 13 chemical measurements). Class labels are intentionally excluded from clustering and evaluation.

**Algorithms:** K-Means, Agglomerative Clustering, DBSCAN. A fixed seed is used for reproducibility.

### 1. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
print('Libraries imported successfully.')

### 2. Load and inspect the dataset
We use the 13 numeric measurements only. The class label is deliberately not used, preventing supervised information from influencing clustering.

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
print('Dataset shape:', X.shape)
display(X.head())
print('Missing values:', X.isna().sum().sum())
display(X.describe().T)

### 3. Exploratory data analysis
Explore feature distributions and correlations. Since measurements use different units and ranges, scale them before applying distance-based algorithms.

In [ ]:
X.hist(figsize=(16, 12), bins=20)
plt.suptitle('Wine Dataset: Feature Distributions', y=1.02)
plt.tight_layout(); plt.show()
plt.figure(figsize=(12, 9))
sns.heatmap(X.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Heatmap'); plt.tight_layout(); plt.show()

### 4. Feature scaling
StandardScaler transforms features to approximately zero mean and unit variance. This prevents high-range measurements from dominating distance calculations.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled matrix shape:', X_scaled.shape)
print('Example scaled row:', np.round(X_scaled[0], 3))

### 5. Select K for K-Means
Compare inertia (elbow method) and silhouette scores for k=2 through 10. We select the K with the highest silhouette in this tested range. This is a heuristic, not proof of the objectively correct number of groups.

In [ ]:
k_values = range(2, 11)
inertias, silhouettes = [], []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(k_values), inertias, marker='o'); ax[0].set_title('K-Means Inertia (Elbow)'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[1].plot(list(k_values), silhouettes, marker='o'); ax[1].set_title('Silhouette vs k'); ax[1].set_xlabel('k'); ax[1].set_ylabel('Silhouette (higher is better)')
plt.tight_layout(); plt.show()
best_k = list(k_values)[int(np.argmax(silhouettes))]
print('Selected k:', best_k)

### 6. Fit clustering algorithms
- **K-Means:** centroid-based, works best with compact/roughly spherical clusters.
- **Agglomerative (Ward):** hierarchical bottom-up merging; Ward linkage minimizes within-cluster variance.
- **DBSCAN:** density-based, can find irregular shapes and label outliers as noise.

K-Means and Agglomerative use the silhouette-selected cluster count. DBSCAN is not forced to use a preset number of groups. All use the same standardized features.

In [ ]:
models = {
    'K-Means': KMeans(n_clusters=best_k, random_state=SEED, n_init=20),
    'Agglomerative (Ward)': AgglomerativeClustering(n_clusters=best_k, linkage='ward'),
    'DBSCAN': DBSCAN(eps=2.2, min_samples=5)
}
labels_by_model = {}
for name, model in models.items():
    labels_by_model[name] = model.fit_predict(X_scaled)
    label_set = set(labels_by_model[name])
    n_clusters = len(label_set - {-1})
    n_noise = int(np.sum(labels_by_model[name] == -1))
    print(f'{name}: clusters={n_clusters}, noise samples={n_noise}')

### 7. Evaluate clustering quality
- **Silhouette:** between −1 and 1; higher is better separation and cohesion.
- **Davies–Bouldin:** lower is better; measures similarity between clusters relative to their spread.
- **Calinski–Harabasz:** higher is better; compares between-cluster and within-cluster dispersion.

DBSCAN's noise points (label −1) are excluded from these quality metrics. Coverage/noise fraction are reported so DBSCAN's scores are not interpreted without considering how many samples it discarded. If a metric cannot be computed, the cell reports NaN.

In [ ]:
def calculate_metrics(X_matrix, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    X_eval, y_eval = X_matrix[keep], labels[keep]
    n_clusters = len(np.unique(y_eval))
    coverage = keep.mean()
    noise_fraction = 1 - coverage
    if n_clusters < 2 or n_clusters >= len(y_eval):
        return {'Clusters': n_clusters, 'Silhouette': np.nan,
                'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan,
                'Coverage': coverage, 'Noise fraction': noise_fraction}
    return {'Clusters': n_clusters,
            'Silhouette': silhouette_score(X_eval, y_eval),
            'Davies-Bouldin': davies_bouldin_score(X_eval, y_eval),
            'Calinski-Harabasz': calinski_harabasz_score(X_eval, y_eval),
            'Coverage': coverage, 'Noise fraction': noise_fraction}
results = pd.DataFrame({name: calculate_metrics(X_scaled, labs)
                       for name, labs in labels_by_model.items()}).T
results = results.sort_values('Silhouette', ascending=False)
display(results.round(4))

### 8. PCA cluster visualization
PCA projects the standardized 13-feature data to 2 dimensions for visualization only. Clustering was performed on all features, not the PCA plot. Projection can distort distances; interpret plots alongside metrics.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
print('Variance explained by PCs:', np.round(pca.explained_variance_ratio_, 4))
print('Total 2D variance explained:', round(pca.explained_variance_ratio_.sum(), 4))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, labels) in zip(axes, labels_by_model.items()):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=35, alpha=0.8)
    ax.set_title(name); ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
plt.suptitle('Clustering Results in PCA Space'); plt.tight_layout(); plt.show()

### 9. Cluster sizes and interpretation
Cluster numbers are arbitrary IDs, not rankings. Review group sizes and use original measurement units to inspect average feature values within K-Means groups.

In [ ]:
for name, labels in labels_by_model.items():
    print(f'\n{name} cluster counts:')
    display(pd.Series(labels).value_counts().sort_index().rename('Samples').to_frame())
kmeans_labels = labels_by_model['K-Means']
profile = X.assign(Cluster=kmeans_labels).groupby('Cluster').mean()
display(profile.round(2))
scaled_profile = pd.DataFrame(X_scaled, columns=X.columns).assign(Cluster=kmeans_labels).groupby('Cluster').mean()
plt.figure(figsize=(14, 6))
sns.heatmap(scaled_profile, cmap='coolwarm', center=0, annot=False)
plt.title('K-Means Profiles (Mean Feature Values in Standard Deviations)')
plt.ylabel('Cluster'); plt.tight_layout(); plt.show()

### 10. Comparative rankings
Do not choose a model from one score alone. Compare three metric rankings, cluster count, and DBSCAN coverage. Average rank is a simple summary (rank 1 is best), not a substitute for interpretation.

In [ ]:
print('Silhouette: higher is better')
display(results.sort_values('Silhouette', ascending=False)[['Silhouette', 'Clusters', 'Coverage', 'Noise fraction']].round(4))
print('\nDavies-Bouldin: lower is better')
display(results.sort_values('Davies-Bouldin')[['Davies-Bouldin', 'Clusters', 'Coverage', 'Noise fraction']].round(4))
print('\nCalinski-Harabasz: higher is better')
display(results.sort_values('Calinski-Harabasz', ascending=False)[['Calinski-Harabasz', 'Clusters', 'Coverage', 'Noise fraction']].round(4))
ranks = pd.DataFrame(index=results.index)
ranks['Silhouette rank'] = results['Silhouette'].rank(ascending=False)
ranks['DB index rank'] = results['Davies-Bouldin'].rank(ascending=True)
ranks['CH rank'] = results['Calinski-Harabasz'].rank(ascending=False)
ranks['Average rank'] = ranks.mean(axis=1)
display(ranks.sort_values('Average rank').round(2))

### 11. Critical analysis and discussion
**Write your observations after running the notebook, using actual outputs:**
1. Identify the algorithm with the highest silhouette and interpret its score.
2. Identify the lowest Davies–Bouldin result and discuss what it indicates.
3. Compare the Calinski–Harabasz ranking; explain any disagreement among metrics.
4. Report DBSCAN cluster count and noise fraction. Is it useful given its coverage?
5. Compare PCA visuals to quantitative results. Do the visual groupings agree?
6. Describe K-Means cluster profiles using a few features from the table; do not treat clusters as causal or as quality grades.

**Limitations:** Results depend on scaling and distance metric. K-Means and Ward tend to favor compact clusters. DBSCAN is sensitive to `eps` and `min_samples` and may label many points noise. PCA omits dimensions. This dataset is relatively small and may not represent all wines.

**Possible improvements:** test multiple DBSCAN parameter values; assess stability across random seeds; try alternative linkages/distances; use metrics cautiously; validate clusters with domain knowledge.

### 12. Ethics, responsible use, and reflection
Clustering is exploratory. Biased, incomplete, or non-representative data may create misleading groups. Cluster assignments are not ground truth and should not drive high-stakes decisions without validation. Document data, settings, and limitations.

**Personal reflection — fill in after running:**
- What did I learn about centroid, hierarchical, and density clustering?
- Which output surprised me, and what evidence explains it?
- Which preprocessing or parameter choice affected results most?
- What would I investigate next?

Replace these prompts with your own concise, experiment-based reflection before submitting.

### 13. Conclusion
Clustering quality is multi-dimensional. A useful solution combines separation and cohesion, adequate coverage, interpretable profiles, and robustness. This assignment applies three clustering methods to a real dataset and compares them using quantitative metrics and visual inspection.

In [ ]:
# Optional: save metric table for report
results.round(4).to_csv('clustering_metrics_comparison.csv', index=True)
print('Saved clustering_metrics_comparison.csv')